# CardioSense — Quick Colab Setup (about 1–2 minutes)

Use this instead of `00_colab_setup.ipynb` in every new Colab session.
It only mounts Drive, gets the code and installs the packages Colab lacks.

It does **not** download any dataset, ask for `kaggle.json`, scan Drive or run the
test suite. Those steps in the full setup notebook are what take 30+ minutes,
and they only ever need to run once (the data stays on Drive).

`04_ecg_feature_fusion.ipynb` does not need this notebook at all; it sets itself up.

In [ ]:
import os, sys, subprocess
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = Path("/content/CardioSense")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/artrai73/CardioSense.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

os.environ["CARDIOSENSE_DATA_ROOT"] = "/content/drive/MyDrive/CardioSense/data"
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
print("Repo:", REPO_DIR, "| data root:", os.environ["CARDIOSENSE_DATA_ROOT"])

In [ ]:
# Installs only what Colab does not already ship (torch is deliberately left alone).
!pip install -q -r requirements-colab.txt
!pip install -q --no-deps -e .

import torch
print("torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# Instant check: looks for one file per dataset, no folder scans.
data = Path(os.environ["CARDIOSENSE_DATA_ROOT"])
checks = {
    "PTB-XL (ECG)":         data / "ecg" / "ptbxl" / "ptbxl_database.csv",
    "ECG waveform cache":   data / "ecg" / "cache",
    "ChestX-ray14 (X-ray)": data / "xray" / "nih" / "Data_Entry_2017_v2020.csv",
}
for name, path in checks.items():
    print(f"{name:22s}: {'found' if path.exists() else 'missing'}")
print()
print("Missing data? Run only that dataset's section (7a/7b/7c) of 00_colab_setup.ipynb, once.")